# Fase 11 — Investigación en profundidad: clasificación, calendario y diagnóstico de Afición

**Requisito previo:** base de datos `golazo_growup` cargada (`python -m src.cargar_datos`).

Convención usada en todo el notebook: los **'días fuertes de audiencia'** son los 2 días de la semana con más vistas medias en `evolucion_diaria` (30 días reales). Es una muestra pequeña (~4 observaciones por día de la semana) — se trata como el mejor proxy disponible, no como un patrón verificado a largo plazo.

## 0. Carga de datos y columnas derivadas

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine

from src import db_connection as dbc

url = dbc.DATABASE_URL or (
    f"postgresql+psycopg2://{dbc.DB_CONFIG['user']}:{dbc.DB_CONFIG['password']}"
    f"@{dbc.DB_CONFIG['host']}:{dbc.DB_CONFIG['port']}/{dbc.DB_CONFIG['dbname']}"
)
if url.startswith('postgresql://'):
    url = url.replace('postgresql://', 'postgresql+psycopg2://', 1)
engine = create_engine(url)

video = pd.read_sql('SELECT * FROM video', engine, parse_dates=['fecha_publicacion'])
retencion = pd.read_sql('SELECT * FROM retencion_audiencia', engine)
evolucion = pd.read_sql('SELECT * FROM evolucion_diaria', engine, parse_dates=['fecha'])

ORDEN_DIAS = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
HOY = pd.Timestamp.now().normalize()

evolucion['dia_semana'] = evolucion['fecha'].dt.day_name()
evolucion['suscriptores_netos'] = evolucion['subscribers_gained'] - evolucion['subscribers_lost']

UMBRAL_SHORT_SEGUNDOS = 180
video['es_short'] = video['duracion_segundos'] <= UMBRAL_SHORT_SEGUNDOS
video['formato'] = video['es_short'].map({True: 'Short', False: 'Largo'})
video['dia_semana_publicacion'] = video['fecha_publicacion'].dt.day_name()

# Antigüedad: se recalcula en cada ejecución contra HOY, así que se actualiza sola día a día
video['antiguedad_dias'] = (HOY - video['fecha_publicacion']).dt.days.clip(lower=1)
video['views_por_dia'] = video['views_totales'] / video['antiguedad_dias']

# Umbral mínimo de antigüedad para usar 'views_por_dia' de forma fiable: un vídeo con 1-2 días
# de vida puede dar ratios absurdos (denominador casi cero) que no reflejan una velocidad real,
# solo ruido de publicación reciente. Se exige al menos 7 días antes de fiarse de su velocidad.
MIN_DIAS_PARA_VELOCIDAD = 7
video['apto_velocidad'] = video['antiguedad_dias'] >= MIN_DIAS_PARA_VELOCIDAD

video['ratio_likes_vista'] = video['likes'] / video['views_totales']
video['ratio_comentarios_vista'] = video['comentarios'] / video['views_totales']
video['z_views_categoria_formato'] = video.groupby(['categoria', 'formato'])['views_totales'].transform(
    lambda s: (s - s.mean()) / s.std() if s.std() > 0 else 0
)

retencion_media_por_video = retencion.groupby('video_id')['audience_watch_ratio'].mean().rename('retencion_media')
video = video.merge(retencion_media_por_video, on='video_id', how='left')

patron_semanal_audiencia = evolucion.groupby('dia_semana')['views'].mean().reindex(ORDEN_DIAS)
dias_fuertes = patron_semanal_audiencia.sort_values(ascending=False).head(2).index.tolist()
video['es_dia_fuerte'] = video['dia_semana_publicacion'].isin(dias_fuertes)

print(f'HOY (referencia de antigüedad): {HOY.date()}')
print(f'Días fuertes de audiencia: {dias_fuertes}')
print(f'Vídeos: {len(video)} | Rango de antigüedad: {video["antiguedad_dias"].min()} a {video["antiguedad_dias"].max()} días')

## 1. Clasificación de vídeos por vistas y tiempo publicado

Por cada vídeo: fecha y día de publicación, antigüedad en días (recalculada contra la fecha de hoy cada vez que se ejecuta el notebook), nivel de vistas, y una métrica de velocidad (`views_por_dia`) que separa 'tiene muchas vistas' de 'ha conseguido muchas vistas rápido'.

In [ ]:
video['nivel_views'] = pd.qcut(video['views_totales'], q=4, labels=['Bajo', 'Medio', 'Alto', 'Muy alto'])
print('Vídeos por nivel de vistas:')
display(video['nivel_views'].value_counts().reindex(['Bajo', 'Medio', 'Alto', 'Muy alto']))

columnas_ficha = ['titulo', 'categoria', 'formato', 'fecha_publicacion', 'dia_semana_publicacion',
                   'antiguedad_dias', 'views_totales', 'views_por_dia', 'likes', 'retencion_media', 'nivel_views']

In [ ]:
print('TOP 10 por VISTAS TOTALES (acumulado de toda la vida del vídeo):')
display(video.sort_values('views_totales', ascending=False).head(10)[columnas_ficha].round(2))

In [ ]:
print('TOP 10 por RETENCIÓN MEDIA:')
display(video.sort_values('retencion_media', ascending=False).head(10)[columnas_ficha].round(2))

In [ ]:
print('TOP 10 por LIKES:')
display(video.sort_values('likes', ascending=False).head(10)[columnas_ficha].round(2))

**Nota metodológica:** para el ranking de velocidad se exige un mínimo de 7 días de antigüedad — sin este filtro, un vídeo publicado ayer con pocas vistas puede dar un ratio absurdamente alto solo porque el denominador (días) es casi cero, no porque esté funcionando especialmente bien.

In [ ]:
print('TOP 10 por VELOCIDAD (views_por_dia, solo vídeos con 7+ dias de antiguedad para evitar outliers):')
top10_velocidad = video[video['apto_velocidad']].sort_values('views_por_dia', ascending=False).head(10)
display(top10_velocidad[columnas_ficha].round(2))

In [ ]:
# Comparación clave: ¿los líderes por vistas totales son los mismos que los líderes por velocidad?
top10_views = set(video.sort_values('views_totales', ascending=False).head(10)['video_id'])
top10_vel = set(top10_velocidad['video_id'])
interseccion = top10_views & top10_vel
print(f'Vídeos que están en AMBOS Top 10 (vistas totales Y velocidad): {len(interseccion)} de 10')

antiguedad_top_views = video[video['video_id'].isin(top10_views)]['antiguedad_dias'].mean()
antiguedad_top_vel = video[video['video_id'].isin(top10_vel)]['antiguedad_dias'].mean()
print(f"Antigüedad media del Top 10 por vistas totales: {antiguedad_top_views:.0f} días")
print(f"Antigüedad media del Top 10 por velocidad (ya filtrado >= 7 días): {antiguedad_top_vel:.0f} días")
if antiguedad_top_views > antiguedad_top_vel * 1.5:
    print('\nLECTURA: el Top de vistas totales está dominado por vídeos antiguos que simplemente han tenido '
          'más tiempo para acumular — el Top de velocidad es la referencia más honesta de qué está '
          'funcionando BIEN AHORA, no solo qué es viejo.')

## 2. Calendario: Opinión post-partido (Shorts) y Seguimiento del club (Largo)

Qué día de la semana se publica cada combinación categoría+formato hoy, y qué día habría que tenerlos listos para llegar publicados ANTES de los días fuertes de audiencia.

In [ ]:
opinion_shorts = video[(video['categoria'] == 'Opinión post-partido') & (video['formato'] == 'Short')]
seguimiento_largos = video[(video['categoria'] == 'Seguimiento del club') & (video['formato'] == 'Largo')]

print(f'Opinión post-partido en Shorts: {len(opinion_shorts)} vídeos')
dist_opinion = opinion_shorts['dia_semana_publicacion'].value_counts(normalize=True).mul(100).reindex(ORDEN_DIAS).fillna(0)
display(dist_opinion.round(1))

print(f'\nSeguimiento del club en Largo: {len(seguimiento_largos)} vídeos')
dist_seguimiento = seguimiento_largos['dia_semana_publicacion'].value_counts(normalize=True).mul(100).reindex(ORDEN_DIAS).fillna(0)
display(dist_seguimiento.round(1))

In [ ]:
# Recomendación de calendario: publicar 1-2 días ANTES del día fuerte más próximo,
# para que el vídeo ya esté ganando tracción cuando llegue el pico de audiencia.
indice_dias = {d: i for i, d in enumerate(ORDEN_DIAS)}

for nombre, dist in [('Opinión post-partido (Short)', dist_opinion), ('Seguimiento del club (Largo)', dist_seguimiento)]:
    dia_actual_top = dist.idxmax()
    print(f"\n{nombre}: día actual más usado para publicar -> {dia_actual_top}")
    for dia_fuerte in dias_fuertes:
        idx_fuerte = indice_dias[dia_fuerte]
        dia_recomendado = ORDEN_DIAS[(idx_fuerte - 2) % 7]
        print(f"  Para llegar con tracción al día fuerte '{dia_fuerte}': tener el vídeo listo/publicado "
              f"para el '{dia_recomendado}' (2 días antes).")

## 3. Shorts de 'Opinión post-partido' vs. otras categorías (Shorts) — día fuerte vs. resto

In [ ]:
shorts = video[video['formato'] == 'Short'].copy()
shorts['es_opinion'] = shorts['categoria'] == 'Opinión post-partido'

comparativa_opinion = shorts.groupby(['es_opinion', 'es_dia_fuerte'])[
    ['views_totales', 'likes', 'ratio_likes_vista', 'retencion_media']
].mean()
comparativa_opinion.index = comparativa_opinion.index.set_names(['Es Opinión', 'Día fuerte'])
print('Shorts — Opinión post-partido vs. resto, cruzado con día fuerte / resto de la semana:')
display(comparativa_opinion.round(3))

In [ ]:
# Lecturas explícitas de las 4 celdas del cruce
try:
    op_fuerte = comparativa_opinion.loc[(True, True)]
    op_resto = comparativa_opinion.loc[(True, False)]
    otras_fuerte = comparativa_opinion.loc[(False, True)]
    otras_resto = comparativa_opinion.loc[(False, False)]

    print(f"Opinión (Short) en día fuerte vs. resto de la semana: "
          f"{op_fuerte['views_totales']:.0f} vs. {op_resto['views_totales']:.0f} vistas medias "
          f"({'MEJOR' if op_fuerte['views_totales'] > op_resto['views_totales'] else 'PEOR'} en día fuerte)")
    print(f"Opinión (Short) vs. otras categorías (Short), en día fuerte: "
          f"{op_fuerte['views_totales']:.0f} vs. {otras_fuerte['views_totales']:.0f} vistas medias "
          f"({'Opinión SUPERA' if op_fuerte['views_totales'] > otras_fuerte['views_totales'] else 'Opinión NO supera'} al resto)")
except KeyError as e:
    print(f'Alguna combinación no tiene datos suficientes: {e}')

## 4. Vídeos largos de 'Seguimiento del club' vs. otras categorías (Largos) — día fuerte vs. resto

In [ ]:
largos = video[video['formato'] == 'Largo'].copy()
largos['es_seguimiento'] = largos['categoria'] == 'Seguimiento del club'

comparativa_seguimiento = largos.groupby(['es_seguimiento', 'es_dia_fuerte'])[
    ['views_totales', 'likes', 'ratio_likes_vista', 'retencion_media']
].mean()
comparativa_seguimiento.index = comparativa_seguimiento.index.set_names(['Es Seguimiento', 'Día fuerte'])
print('Largos — Seguimiento del club vs. resto, cruzado con día fuerte / resto de la semana:')
display(comparativa_seguimiento.round(3))

In [ ]:
try:
    seg_fuerte = comparativa_seguimiento.loc[(True, True)]
    seg_resto = comparativa_seguimiento.loc[(True, False)]
    otras_fuerte_l = comparativa_seguimiento.loc[(False, True)]

    print(f"Seguimiento (Largo) en día fuerte vs. resto de la semana: "
          f"{seg_fuerte['views_totales']:.0f} vs. {seg_resto['views_totales']:.0f} vistas medias "
          f"({'MEJOR' if seg_fuerte['views_totales'] > seg_resto['views_totales'] else 'PEOR'} en día fuerte)")
    print(f"Seguimiento (Largo) vs. otras categorías (Largo), en día fuerte: "
          f"{seg_fuerte['views_totales']:.0f} vs. {otras_fuerte_l['views_totales']:.0f} vistas medias "
          f"({'Seguimiento SUPERA' if seg_fuerte['views_totales'] > otras_fuerte_l['views_totales'] else 'Seguimiento NO supera'} al resto)")
except KeyError as e:
    print(f'Alguna combinación no tiene datos suficientes: {e}')

## 5. Vídeos virales: Shorts vs. Largo, comparados con no-virales de su MISMA categoría

Se listan todos los vídeos con `z_views_categoria_formato > 2` (en tu ejecución deberían ser 18).

In [ ]:
picos_virales = video[video['z_views_categoria_formato'] > 2]
print(f'Vídeos virales detectados: {len(picos_virales)}')
display(picos_virales['formato'].value_counts())

columnas_virales = ['titulo', 'categoria', 'formato', 'fecha_publicacion', 'dia_semana_publicacion',
                     'antiguedad_dias', 'views_totales', 'views_por_dia', 'retencion_media']
display(picos_virales[columnas_virales].sort_values('views_por_dia', ascending=False).round(2))

In [ ]:
for formato in ['Short', 'Largo']:
    virales_f = picos_virales[picos_virales['formato'] == formato]
    if len(virales_f) == 0:
        print(f'\nSin virales de tipo {formato}.')
        continue
    print(f"\n--- {formato}s virales (n={len(virales_f)}) vs. no-virales de SU MISMA categoría ---")
    for categoria in virales_f['categoria'].unique():
        viral_cat = virales_f[virales_f['categoria'] == categoria]
        no_viral_cat = video[
            (video['categoria'] == categoria) & (video['formato'] == formato)
            & (video['z_views_categoria_formato'] <= 2)
        ]
        # Mediana en vez de media: views_por_dia es un ratio con denominador pequeño para vídeos
        # recientes, así que unos pocos casos extremos pueden desvirtuar la media por completo.
        print(f"  {categoria}: viral n={len(viral_cat)} | views_por_dia (mediana) "
              f"{viral_cat['views_por_dia'].median():.1f} vs. no-viral {no_viral_cat['views_por_dia'].median():.1f} | "
              f"retención {viral_cat['retencion_media'].mean()*100:.1f}% vs. {no_viral_cat['retencion_media'].mean()*100:.1f}%")

## 6. Diagnóstico de 'Afición': ¿contenido débil o factor externo?

Se descartan explícitamente: tamaño de muestra, mezcla de formato, y antigüedad/novedad antes de concluir nada sobre la calidad del contenido en sí.

In [ ]:
conteo_categorias = video['categoria'].value_counts()
print('Vídeos por categoría:')
display(conteo_categorias)

n_aficion = conteo_categorias.get('Afición', 0)
n_media_resto = conteo_categorias.drop('Afición', errors='ignore').mean()
print(f"\nAfición tiene {n_aficion} vídeos; el resto de categorías tiene de media {n_media_resto:.0f}.")
if n_aficion < n_media_resto * 0.5:
    print('AVISO: Afición tiene bastante menos de la mitad de vídeos que la categoría media — cualquier '
          'diferencia de rendimiento debe leerse con cautela por tamaño de muestra.')

In [ ]:
aficion = video[video['categoria'] == 'Afición']
resto = video[video['categoria'] != 'Afición']

# Descarte 1: mezcla de formato distinta
mix_aficion = aficion['formato'].value_counts(normalize=True).mul(100)
mix_resto = resto['formato'].value_counts(normalize=True).mul(100)
print('Mezcla de formato — Afición vs. resto del canal:')
display(pd.DataFrame({'Afición': mix_aficion, 'Resto': mix_resto}).round(1))

# Descarte 2: antigüedad / novedad
print(f"\nAntigüedad media — Afición: {aficion['antiguedad_dias'].mean():.0f} días | "
      f"Resto: {resto['antiguedad_dias'].mean():.0f} días")

In [ ]:
# Comparación controlada: usando la MEDIANA de views_por_dia (más robusta que la media frente a
# vídeos muy recientes con denominador casi cero), y separado por formato
# (neutraliza el efecto de mezcla Shorts/largos). Se excluyen además los vídeos con menos de 7 días
# de antigüedad (apto_velocidad), que no han tenido tiempo de alcanzar una velocidad estable.
print('Comparación controlada por formato y velocidad (mediana de views_por_dia, no vistas totales crudas):')
for formato in ['Short', 'Largo']:
    a = aficion[(aficion['formato'] == formato) & aficion['apto_velocidad']]
    r = resto[(resto['formato'] == formato) & resto['apto_velocidad']]
    if len(a) == 0 or len(r) == 0:
        print(f'  {formato}: datos insuficientes en Afición o en el resto para comparar.')
        continue
    print(f"  {formato} — Afición (n={len(a)}): {a['views_por_dia'].median():.2f} vistas/día (mediana), "
          f"retención {a['retencion_media'].mean()*100:.1f}% | "
          f"Resto (n={len(r)}): {r['views_por_dia'].median():.2f} vistas/día (mediana), "
          f"retención {r['retencion_media'].mean()*100:.1f}%")

In [ ]:
# Veredicto final, tras los descartes anteriores — con mediana y filtro de antigüedad mínima
aficion_apta = aficion[aficion['apto_velocidad']]
resto_apto = resto[resto['apto_velocidad']]
gap_velocidad = aficion_apta['views_por_dia'].median() / resto_apto['views_por_dia'].median() - 1
gap_retencion = aficion['retencion_media'].mean() - resto['retencion_media'].mean()

print(f"Gap de velocidad (mediana, Afición vs. resto, ya controlado por antigüedad): {gap_velocidad*100:+.1f}%")
print(f"Gap de retención: {gap_retencion*100:+.1f} puntos porcentuales")

if n_aficion < n_media_resto * 0.5:
    print('\nVEREDICTO: con este tamaño de muestra, cualquier conclusión sobre \'el contenido en sí\' es '
          'poco fiable — antes de decidir nada, GrowUP necesitaría más vídeos de Afición para confirmar el '
          'patrón, no solo estos pocos.')
elif gap_velocidad < -0.15 and gap_retencion < -0.03:
    print('\nVEREDICTO: incluso controlando por formato, antigüedad y usando la mediana (robusta a outliers), '
          'Afición sigue por debajo en velocidad Y retención — el patrón apunta al contenido en sí, no a un '
          'artefacto de los datos.')
else:
    print('\nVEREDICTO: la diferencia se explica en buena parte por formato/antigüedad/outliers puntuales, '
          'no por el contenido en sí — no hay evidencia sólida de que Afición \'funcione peor\' de forma inherente.')

## 7. Vídeos con vistas anormalmente altas: ¿en qué clasificaciones caen, y qué pasó con los suscriptores?

In [ ]:
columnas_cruce = ['titulo', 'categoria', 'formato', 'fecha_publicacion', 'dia_semana_publicacion', 'nivel_views']
print('Vídeos con vistas anormalmente altas (mismos del punto 5) — clasificación del punto 1:')
display(picos_virales[columnas_cruce])

top10_views_ids = set(video.sort_values('views_totales', ascending=False).head(10)['video_id'])
top10_vel_ids = set(video[video['apto_velocidad']].sort_values('views_por_dia', ascending=False).head(10)['video_id'])
top10_retencion_ids = set(video.sort_values('retencion_media', ascending=False).head(10)['video_id'])

for _, v in picos_virales.iterrows():
    etiquetas = []
    if v['video_id'] in top10_views_ids: etiquetas.append('Top10 vistas totales')
    if v['video_id'] in top10_vel_ids: etiquetas.append('Top10 velocidad')
    if v['video_id'] in top10_retencion_ids: etiquetas.append('Top10 retención')
    print(f"  {v['titulo'][:40]:40s} -> {', '.join(etiquetas) if etiquetas else '(ninguna clasificación Top10)'}")

In [ ]:
# Cruce con ganancia neta de suscriptores: desde el día de publicación hasta 3 días después
# Solo es posible para vídeos cuya fecha de publicación cae dentro de la ventana de evolucion_diaria (30 días)
fecha_min_evolucion = evolucion['fecha'].min()
fecha_max_evolucion = evolucion['fecha'].max()

resultados_cruce = []
for _, v in picos_virales.iterrows():
    fecha_pub = v['fecha_publicacion'].normalize()
    if fecha_pub < fecha_min_evolucion or fecha_pub > fecha_max_evolucion:
        resultados_cruce.append({
            'titulo': v['titulo'], 'fecha_publicacion': fecha_pub.date(),
            'dia_semana_publicacion': v['dia_semana_publicacion'],
            'suscriptores_netos_3dias': None, 'nota': 'fuera de la ventana de 30 días de evolucion_diaria',
        })
        continue
    ventana = evolucion[
        (evolucion['fecha'] >= fecha_pub) & (evolucion['fecha'] <= fecha_pub + pd.Timedelta(days=3))
    ]
    resultados_cruce.append({
        'titulo': v['titulo'], 'fecha_publicacion': fecha_pub.date(),
        'dia_semana_publicacion': v['dia_semana_publicacion'],
        'suscriptores_netos_3dias': int(ventana['suscriptores_netos'].sum()) if len(ventana) else None,
        'nota': f'{len(ventana)} día(s) disponibles en la ventana' if len(ventana) else 'sin datos en la ventana',
    })

df_cruce = pd.DataFrame(resultados_cruce)
display(df_cruce)

con_dato = df_cruce[df_cruce['suscriptores_netos_3dias'].notna()]
if len(con_dato):
    media_normal = evolucion['suscriptores_netos'].mean() * 4  # equivalente a 4 días de referencia (día pub + 3 después)
    print(f"\nMedia de suscriptores netos en una ventana de 4 días cualquiera (referencia): {media_normal:.1f}")
    print(f"Media real tras estos vídeos virales: {con_dato['suscriptores_netos_3dias'].mean():.1f}")
else:
    print('\nNinguno de los vídeos virales cae dentro de la ventana de 30 días de evolucion_diaria — '
          'no se puede hacer el cruce con datos disponibles actualmente. Sería necesario ampliar el rango '
          'de evolucion_diaria (Fase 4) para poder analizar el impacto en suscriptores de vídeos antiguos.')